In [1]:

from QuantLab.alpha import compute_all_alphas, save_alpha_results
from QuantLab.frs import save_frs_results, ETFS
from QuantLab.signal.compute_signal import save_signal_results
from QuantLab.utils import load_pathes, load_panel, save_panel_as_bars, save_non_etf_bars, init_db
from pathlib import Path
import pandas as pd

In [2]:
pathes    = load_pathes()

#rename a new folder which will not be pushed to github
target_folder = "data"

save_path = pathes["ROOT"] / target_folder 
save_path.mkdir(parents=True, exist_ok=True)

db_path = save_path / "datapool.db"

In [3]:
# ── Cell 3: Load panel data ───────────────────────────────────────────────────
processed_dir = pathes["data"] / "processed"
data_path = processed_dir / "data.csv"

panel = load_panel(data_path)
print(f"Panel loaded: {len(panel['close'])} dates x {len(panel['close'].columns)} tickers")

Panel loaded: 1551 dates x 11 tickers


In [4]:
# ── Cell 4: Init SQLite DB + write bars ──────────────────────────────────────
conn = init_db(db_path)
save_panel_as_bars(panel, conn)          # ETF bars (11 tickers)
save_non_etf_bars(data_path, conn)       # Benchmark + Index bars (SPY/SPX/VIX/USGG10YR)

Database ready: E:\CUHK\trimester3\practicum\LAB\data\datapool.db
Assets seeded: 11 ETF, 2 Benchmark, 2 Index, 5 Macro → 20 total
ETF bars saved: 17061 daily, 3575 weekly rows → SQLite
Non-ETF bars appended: ['SPY', 'SPX', 'VIX', 'T10Y2Y', 'BAMLH0A0HYM2', 'DTWEXBGS', 'DCOILWTICO', 'T10YIE', 'USGG10YR'] → SQLite


In [5]:
# ── Cell 5: Compute all alphas → write to SQLite weekly_alpha ────────────────
results        = compute_all_alphas(panel)
alpha_combined = save_alpha_results(results, conn)

Saved: 1889657 rows to weekly_alpha, 118 alphas to alpha table


In [6]:
# ── Cell 6: Compute FRS → write to SQLite weekly_frs ─────────────────────────
save_frs_results(conn, etfs=ETFS)

  XLB: 321 rows
  XLC: 321 rows
  XLE: 321 rows
  XLF: 321 rows
  XLI: 321 rows
  XLK: 321 rows
  XLP: 321 rows
  XLU: 321 rows
  XLV: 321 rows
  XLRE: 321 rows
  XLY: 321 rows

FRS complete → SQLite weekly_frs table (10593 rows, long format)


In [7]:
# ── Cell 7: Compute Signals → write to SQLite weekly_signal ───────────────────
save_signal_results(conn, etfs=ETFS)

e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn

Signal complete → SQLite weekly_signal table (signals=6)


e:\CUHK\trimester3\practicum\LAB\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(


In [8]:
# ── Cell 7: Close connection ──────────────────────────────────────────────────
conn.close()
print("Done. Database:", db_path)

Done. Database: E:\CUHK\trimester3\practicum\LAB\data\datapool.db
